In [21]:
import os
import yaml
import json
import argparse
from diambra.arena import load_settings_flat_dict, SpaceTypes
from diambra.arena.stable_baselines3.make_sb3_env import EnvironmentSettings, WrappersSettings, RecordingSettings
from make_sb3_env import make_sb3_env
from diambra.arena.stable_baselines3.sb3_utils import linear_schedule, AutoSave
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList
from wandb.integration.sb3 import WandbCallback
import wandb
from entropy_decay import EntropyDecayCallback
# diambra run -s 8 python stable_baselines3/training.py --cfgFile $PWD/stable_baselines3/cfg_files/sfiii3n/sr6_128x4_das_nc.yaml
import datetime

def main(cfg_file, model_checkpoint, num_envs=1):
    # Read the cfg file
    yaml_file = open(cfg_file)
    params = yaml.load(yaml_file, Loader=yaml.FullLoader)
    print("Config parameters = ", json.dumps(params, sort_keys=True, indent=4))
    yaml_file.close()
    port = os.environ["BASE_PORT"]
    if not port.isnumeric():
        raise Exception("ERROR: BASE_PORT environment variable must be set to a number.")
    base_port = int(port)
    os.environ["DIAMBRA_ENVS"] = " ".join([f"localhost:{base_port + i}" for i in range(num_envs)])

    # Settings
    params["settings"]["action_space"] = SpaceTypes.DISCRETE if params["settings"]["action_space"] == "discrete" else SpaceTypes.MULTI_DISCRETE
    settings = load_settings_flat_dict(EnvironmentSettings, params["settings"])

    # Wrappers Settings
    wrappers_settings = load_settings_flat_dict(WrappersSettings, params["wrappers_settings"])

    # Create environment
    record_settings = RecordingSettings()
    
    env, num_envs = make_sb3_env(settings.game_id, settings, wrappers_settings,use_subprocess=False, episode_recording_settings=record_settings)
    print("Activated {} environment(s)".format(num_envs))

    # Policy param
    policy_kwargs = params["policy_kwargs"]

    # PPO settings
    ppo_settings = params["ppo_settings"]
    gamma = ppo_settings["gamma"]

    learning_rate = linear_schedule(ppo_settings["learning_rate"][0], ppo_settings["learning_rate"][1])
    clip_range = linear_schedule(ppo_settings["clip_range"][0], ppo_settings["clip_range"][1])
    clip_range_vf = clip_range
    batch_size = ppo_settings["batch_size"]
    n_epochs = ppo_settings["n_epochs"]
    n_steps = ppo_settings["n_steps"]
    ent_coef = ppo_settings["ent_coef"]
    if "target_kl" in ppo_settings:
        target_kl = ppo_settings["target_kl"]
    else:
        target_kl = None

    import random
    # Returns an integer (seconds) - generally preferred for storage

    
    agent = PPO.load(model_checkpoint, env=env, ent_coef=ent_coef, target_kl=target_kl,
                         gamma=gamma, learning_rate=learning_rate, clip_range=clip_range,
                         clip_range_vf=clip_range_vf, policy_kwargs=policy_kwargs)

    return env, agent

cfgFile = "config/config_render.yaml"
model_checkpoint="0_autosave_22000000"
os.environ["BASE_PORT"]="32769"
env, agent = main(cfgFile, model_checkpoint, 1)
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv
import gymnasium as gym

INFO:diambra.arena.engine.interface:Trying to connect to DIAMBRA Engine server (timeout=600s)...
INFO:diambra.arena.engine.interface:... done.


Config parameters =  {
    "folders": {
        "model_name": "sr6_128x4_das_nc",
        "parent_dir": "./results/"
    },
    "num_envs": 16,
    "policy_kwargs": {
        "net_arch": [
            64,
            64
        ]
    },
    "ppo_settings": {
        "autosave_freq": 100000,
        "batch_size": 256,
        "clip_range": [
            0.15,
            0.025
        ],
        "ent_coef": 0,
        "gamma": 0.94,
        "learning_rate": [
            0.00025,
            2.5e-06
        ],
        "model_checkpoint": "0",
        "n_epochs": 4,
        "n_steps": 128,
        "time_steps": 50000000
    },
    "settings": {
        "action_space": 2,
        "characters": "Ryu",
        "continue_game": 0.0,
        "difficulty": 6,
        "game_id": "sfiii3n",
        "outfits": 2,
        "step_ratio": 1
    },
    "wrappers_settings": {
        "add_last_action": true,
        "dilation": 6,
        "exclude_image_scaling": true,
        "filter_keys": [
        

INFO:diambra.arena.arena_gym:EnvironmentSettings(game_id='sfiii3n', frame_shape=(0, 0, 0), step_ratio=1, disable_keyboard=True, disable_joystick=True, render_mode='rgb_array', splash_screen=True, rank=0, env_address='localhost:32769', grpc_timeout=600, seed=1761508780, difficulty=6, continue_game=0.0, show_final=False, tower=3, _last_seed=1761508780, pb_model=game_id: "sfiii3n"
frame_shape {
}
step_ratio: 1
n_players: 1
disable_keyboard: true
disable_joystick: true
action_spaces: MULTI_DISCRETE
episode_settings {
}
, n_players=1, action_space=2, role=None, characters='Ryu', outfits=2, super_art=None, fighting_style=None, ultimate_style=None, speed_mode=None)


Activated 1 environment(s)
Wrapping the env in a VecTransposeImage.


In [22]:
def run(model_checkpoint, env, run_int : int):
#     MAX_VIDEO_LENGTH=1000000000
#     video_folder="."
#     env = VecVideoRecorder(
#     env,
#     video_folder,
#     # Starts recording at the very beginning (global step 0)
#     record_video_trigger=lambda step: step == 0, 
#     video_length=MAX_VIDEO_LENGTH, 
#     name_prefix=f"full-episode-"
# )
    from video_saver import StreamingVideoRender
    saver = StreamingVideoRender(save_path=f"full_video_{model_checkpoint}_{run_int}.mp4", fps=60)
    def unwrap(env):
        return env.envs[0].env.env.env.env.env.env.env.env.env.env
    observation = env.reset()
    cumulative_reward = 0
    saver.start(unwrap(env).render().shape)
    while True:
        action, _state = agent.predict(observation, deterministic=False)
        observation, reward, done, info = env.step(action)
        render_env = unwrap(env)
        frame = render_env.render()
        saver.step(frame)
        cumulative_reward += reward
        if (reward != 0):
            print("Cumulative reward =", cumulative_reward)
        if done:
            observation = env.reset()
            break
    saver.stop()
    return cumulative_reward
for i in range(60):
    reward = run(model_checkpoint, env, i).item()
    with open(f"{model_checkpoint}_{i}.json", 'w') as f:
        json.dump({"reward":reward}, f)

Video rendering started: full_video_0_autosave_22000000_0.mp4 (384x224 @ 60 FPS)
Cumulative reward = [-0.17391305]
Cumulative reward = [-0.16149068]
Cumulative reward = [0.18633541]
Cumulative reward = [0.21118014]
Cumulative reward = [0.23602486]
Cumulative reward = [0.26086956]
Cumulative reward = [0.28571427]
Cumulative reward = [0.29813662]
Cumulative reward = [0.31055897]
Cumulative reward = [0.32298133]
Cumulative reward = [0.36024842]
Cumulative reward = [0.37267077]
Cumulative reward = [0.38509312]
Cumulative reward = [0.65838504]
Cumulative reward = [0.8819875]
Cumulative reward = [1.0434783]
Cumulative reward = [1.0559006]
Cumulative reward = [1.0683229]
Cumulative reward = [1.0807452]
Cumulative reward = [1.0931675]
Cumulative reward = [1.1055899]
Cumulative reward = [1.0559005]
Cumulative reward = [1.4037266]
Cumulative reward = [1.5776396]
Cumulative reward = [1.6024843]
Cumulative reward = [1.6273291]
Cumulative reward = [1.7763975]
Cumulative reward = [1.689441]
Cumulati

In [15]:
import glob
z = [(i, json.load(open(i))["reward"]) for i in glob.glob("*.json")]
z.sort(key=lambda x:x[-1])
z

[('0_autosave_32300000_21.json', 3.0310566425323486),
 ('0_autosave_32300000_16.json', 3.478266477584839),
 ('0_autosave_32300000_18.json', 3.503103256225586),
 ('0_autosave_32300000_15.json', 5.71428918838501),
 ('0_autosave_32300000_7.json', 5.975159168243408),
 ('0_autosave_32300000_32.json', 6.372685432434082),
 ('0_autosave_32300000_1.json', 7.018640995025635),
 ('0_autosave_32300000_24.json', 7.05590295791626),
 ('0_autosave_32300000_3.json', 7.515536308288574),
 ('0_autosave_32300000_28.json', 7.652174472808838),
 ('0_autosave_32300000_11.json', 7.689452171325684),
 ('0_autosave_32300000_0.json', 7.776401996612549),
 ('0_autosave_32300000_27.json', 7.77640962600708),
 ('0_autosave_32300000_19.json', 7.937901020050049),
 ('0_autosave_32300000_10.json', 8.59628963470459),
 ('0_autosave_32300000_6.json', 8.695652961730957),
 ('0_autosave_32300000_12.json', 8.83230972290039),
 ('0_autosave_32300000_22.json', 8.919269561767578),
 ('0_autosave_32300000_9.json', 9.515547752380371),
 ('

In [17]:
[i[1] for i in z]

[3.0310566425323486,
 3.478266477584839,
 3.503103256225586,
 5.71428918838501,
 5.975159168243408,
 6.372685432434082,
 7.018640995025635,
 7.05590295791626,
 7.515536308288574,
 7.652174472808838,
 7.689452171325684,
 7.776401996612549,
 7.77640962600708,
 7.937901020050049,
 8.59628963470459,
 8.695652961730957,
 8.83230972290039,
 8.919269561767578,
 9.515547752380371,
 10.409963607788086,
 10.472067832946777,
 10.509332656860352,
 10.534168243408203,
 10.633552551269531,
 10.645971298217773,
 11.540388107299805,
 11.714300155639648,
 11.788830757141113,
 12.099394798278809,
 13.329200744628906,
 13.478273391723633,
 15.32919979095459,
 16.04970359802246]

In [6]:
import cv2
import numpy as np

# 1. Define the FourCC code for H.264 (avc1)
fourcc = cv2.VideoWriter_fourcc(*'avc1') 

# 2. Set the output file name, frame size, and frames per second (FPS)
output_filename = 'output_video.mp4'
frame_width = 640
frame_height = 480
fps = 30.0

# 3. Initialize the VideoWriter object
out = cv2.VideoWriter(
    output_filename, 
    fourcc, 
    fps, 
    (frame_width, frame_height)
)

# Check if the VideoWriter was successfully initialized
if not out.isOpened():
    print("Error: Could not open the output video file for writing.")
else:
    # 4. Write some frames (e.g., 90 frames for 3 seconds of video)
    print(f"Writing {90} frames to {output_filename}...")
    for i in range(90):
        # Create a blank frame (e.g., black)
        frame = np.zeros((frame_height, frame_width, 3), dtype=np.uint8) 
        
        # Add a colored square or text to show it's working
        cv2.putText(frame, f"Frame: {i}", (50, 240), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Write the frame
        out.write(frame)

    # 5. Release the VideoWriter object
    out.release()
    print("Video writing complete and resources released.")

Error: Could not open the output video file for writing.


[ERROR:0@259.536] global cap_ffmpeg_impl.hpp:3207 open Could not find encoder for codec_id=27, error: Encoder not found
[ERROR:0@259.536] global cap_ffmpeg_impl.hpp:3285 open VIDEOIO/FFMPEG: Failed to initialize VideoWriter


In [7]:
from diambra.arena.utils.diambra_data_loader import DiambraDataLoader
dataset_path = "videos/"
data_loader = DiambraDataLoader(dataset_path)

n_loops = data_loader.reset()
from video_saver import StreamingVideoRender

saver = None
terminated = truncated = False
while not (terminated or truncated):
    observation, action, reward, terminated, truncated, info = data_loader.step()
    frame = observation["frame"]
    if saver is None:
        saver = StreamingVideoRender(save_path=f"full_video_{model_checkpoint}.mp4", fps=60)
        saver.start(frame.shape)
    saver.step(frame)
saver.stop()

Renderer is already stopped.


FileNotFoundError: The path 'videos/' does not exist.

In [2]:
from stable_baselines3.common.evaluation import evaluate_policy
mean_reward, std_reward = evaluate_policy(agent, agent.get_env(), n_eval_episodes=4, deterministic=False, return_episode_rewards=True)

In [4]:
mean_reward, std_reward

(np.float64(3.8913045), np.float64(1.256268050068237))

In [5]:
print("Reward: {} (avg) ± {} (std)".format(mean_reward, std_reward))

# Run trained agent
observation = env.reset()
cumulative_reward = 0
while True:

    action, _state = agent.predict(observation, deterministic=False)
    observation, reward, done, info = env.step(action)

    cumulative_reward += reward
    if (reward != 0):
        print("Cumulative reward =", cumulative_reward)

    if done:
        observation = env.reset()
        break

Reward: 3.8913045 (avg) ± 1.256268050068237 (std)


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()